# IVF ANN Backend: Recall vs. Speed
Vector Engine's pure-numpy `ivf` backend, compared against exact `bruteforce` search. No FAISS required — this runs anywhere, including hosts without a `faiss-cpu` wheel (e.g. macOS arm64).

In [ ]:
%pip install -q vector-engine

## Build a synthetic dataset and an exact baseline index

In [ ]:
import numpy as np

from vector_engine import VectorArray, VectorIndex

rng = np.random.default_rng(7)
n, d, nq, k = 20000, 128, 200, 10

xb = VectorArray.from_numpy(rng.standard_normal((n, d)).astype(np.float32), ids=np.arange(n))
xq = VectorArray.from_numpy(rng.standard_normal((nq, d)).astype(np.float32), ids=np.arange(nq))

exact_index = VectorIndex.create(xb, metric="l2", backend="bruteforce")
exact_result = exact_index.search(xq, k=k)
print("exact backend ready:", exact_index.runtime_stats())

## Build an IVF index and sweep `nprobe`
`n_clusters` controls how finely the index is partitioned at build time. `nprobe` controls how many of those clusters get scanned per query — the recall/speed knob.

In [ ]:
import time


def recall_at_k(exact_ids, other_ids):
    hits, total = 0, 0
    for row_exact, row_other in zip(exact_ids, other_ids):
        hits += len(set(row_exact.tolist()) & set(row_other.tolist()))
        total += len(row_exact)
    return hits / total


def time_search(index, queries, k, loops=5):
    start = time.perf_counter()
    for _ in range(loops):
        index.search(queries, k=k)
    elapsed = time.perf_counter() - start
    return (queries.shape[0] * loops) / elapsed


rows = []
for nprobe in (1, 4, 8, 16, 32, 100):
    ivf_index = VectorIndex.create(
        xb, metric="l2", backend="ivf",
        backend_config={"n_clusters": 100, "nprobe": nprobe, "random_state": 7},
    )
    ivf_result = ivf_index.search(xq, k=k)
    rows.append({
        "nprobe": nprobe,
        "recall_at_10": round(recall_at_k(exact_result.ids, ivf_result.ids), 3),
        "qps": round(time_search(ivf_index, xq, k), 1),
    })

for row in rows:
    print(row)

## Reading the tradeoff

- Low `nprobe` scans few clusters: fast, but recall drops sharply.
- `nprobe == n_clusters` scans everything and matches `bruteforce` on recall (1.0), but the current implementation scores clusters with a per-query loop, so it is *slower* than `bruteforce`'s single vectorized matmul at that point — a known, documented tradeoff (see `docs/releases/v1.2.0.md`), not a bug.
- The useful range for `ivf` today is low-to-moderate `nprobe`, where it is meaningfully faster than `bruteforce` and recall is acceptable for the use case.

Prefer the `faiss` backend when available for the best recall/speed tradeoff. `ivf` exists for hosts where `faiss-cpu` cannot be installed at all.